# Clasificación del uso de mascarilla con CNN

**Jose Eduardo Perez Abundiz · A01666860**  
Instituto Tecnológico y de Estudios Superiores de Monterrey, Campus Querétaro  
Inteligencia artificial avanzada para la ciencia de datos (Gpo 601) · Dr. Benjamín Valdés

Este notebook reproduce en **Google Colab** el ETL y el entrenamiento de dos redes convolucionales (CNN) sin transferencia de aprendizaje. La entrada es un rostro previamente recortado; la salida es una de tres clases: `with_mask`, `without_mask` o `mask_weared_incorrect`. No se detectan rostros automáticamente en fotografías completas.

**Uso:** abre este archivo en Colab y elige *Entorno de ejecución → Ejecutar todo*. La descarga del dataset requiere internet y el entrenamiento puede tardar varios minutos. También funciona desde la carpeta local del proyecto si ya existe `data/raw`. El conjunto de prueba queda reservado: este notebook no calcula métricas sobre él.

Fuente: [Face Mask Detection (Kaggle)](https://www.kaggle.com/datasets/andrewmvd/face-mask-detection/data). El reporte de resultados está en `reports/reporte_deep_learning_mascarillas.pdf` del proyecto.

## 1. Entorno y descarga

En Colab se descarga el conjunto público con `kagglehub`; en el equipo local se reutilizan las imágenes y XML ya guardados. Se comprueba que haya 853 pares antes de preparar los recortes. El notebook crea sus archivos de salida dentro de `/content/face_mask_detection` en Colab.

In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys

try:
    import google.colab  # Disponible únicamente en Colab
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

BASE = Path('/content/face_mask_detection') if EN_COLAB else (Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd())
BASE.mkdir(parents=True, exist_ok=True)
(BASE / 'outputs/figuras').mkdir(parents=True, exist_ok=True)

for modulo, paquete in [('tensorflow', 'tensorflow'), ('pandas', 'pandas'), ('sklearn', 'scikit-learn'), ('PIL', 'Pillow'), ('matplotlib', 'matplotlib')]:
    if importlib.util.find_spec(modulo) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', paquete])

IMAGENES = BASE / 'data/raw/images'
ANOTACIONES = BASE / 'data/raw/annotations'
SALIDA = BASE / 'data/processed'

if not (IMAGENES.is_dir() and ANOTACIONES.is_dir()):
    if importlib.util.find_spec('kagglehub') is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'kagglehub'])
    import kagglehub
    descarga = Path(kagglehub.dataset_download('andrewmvd/face-mask-detection'))
    candidatas_imagenes = [p for p in descarga.rglob('images') if p.is_dir()]
    candidatas_xml = [p for p in descarga.rglob('annotations') if p.is_dir()]
    if len(candidatas_imagenes) != 1 or len(candidatas_xml) != 1:
        raise FileNotFoundError('No se encontraron las carpetas images y annotations del dataset.')
    IMAGENES, ANOTACIONES = candidatas_imagenes[0], candidatas_xml[0]

print('Carpeta de trabajo:', BASE)
print('Imágenes:', len(list(IMAGENES.glob('*.png'))))
print('XML:', len(list(ANOTACIONES.glob('*.xml'))))
assert len(list(IMAGENES.glob('*.png'))) == 853
assert len(list(ANOTACIONES.glob('*.xml'))) == 853

## 2. ETL: de fotografías a rostros

Cada XML señala los rostros de una fotografía y su clase. Se ajustan las cajas que exceden un píxel el borde, se descartan los rostros menores de 10 píxeles en alguna dimensión y se convierten los recortes a RGB de 128 × 128. La división se hace por **fotografía original** antes de guardar los recortes; así, rostros de una misma foto no quedan en conjuntos diferentes. La distribución aproximada es 70 % entrenamiento, 15 % validación y 15 % prueba.

Esta celda replica `preparar_dataset.py`. Si los 3,603 recortes ya están preparados, los reutiliza.

In [ ]:
import xml.etree.ElementTree as ET
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split

CLASES = {'with_mask', 'without_mask', 'mask_weared_incorrect'}

def leer_anotaciones():
    registros = []

    for ruta_xml in sorted(ANOTACIONES.glob("*.xml")):
        raiz = ET.parse(ruta_xml).getroot()
        nombre_imagen = raiz.findtext("filename")
        ruta_imagen = IMAGENES / nombre_imagen

        if not ruta_imagen.exists():
            raise FileNotFoundError(ruta_imagen)

        ancho_xml = int(raiz.findtext("size/width"))
        alto_xml = int(raiz.findtext("size/height"))

        with Image.open(ruta_imagen) as imagen:
            if imagen.size != (ancho_xml, alto_xml):
                raise ValueError(
                    f"Dimensiones distintas en {nombre_imagen}"
                )

        for numero, objeto in enumerate(
            raiz.findall("object"), start=1
        ):
            clase = objeto.findtext("name")

            if clase not in CLASES:
                raise ValueError(
                    f"Clase desconocida en {ruta_xml.name}: {clase}"
                )

            caja = objeto.find("bndbox")
            xmin = int(caja.findtext("xmin"))
            ymin = int(caja.findtext("ymin"))
            xmax = int(caja.findtext("xmax"))
            ymax = int(caja.findtext("ymax"))

            # Ajustar la caja a los límites reales de la imagen.
            xmin_usado = max(0, xmin)
            ymin_usado = max(0, ymin)
            xmax_usado = min(ancho_xml, xmax)
            ymax_usado = min(alto_xml, ymax)

            ancho_rostro = xmax_usado - xmin_usado
            alto_rostro = ymax_usado - ymin_usado

            if ancho_rostro <= 0 or alto_rostro <= 0:
                raise ValueError(
                    f"Caja sin área en {ruta_xml.name}, objeto {numero}"
                )

            registros.append({
                "imagen_original": nombre_imagen,
                "numero_objeto": numero,
                "clase": clase,
                "xmin_original": xmin,
                "ymin_original": ymin,
                "xmax_original": xmax,
                "ymax_original": ymax,
                "xmin_usado": xmin_usado,
                "ymin_usado": ymin_usado,
                "xmax_usado": xmax_usado,
                "ymax_usado": ymax_usado,
                "ancho_rostro": ancho_rostro,
                "alto_rostro": alto_rostro,
            })

    return pd.DataFrame(registros)


def dividir_por_imagen(rostros):
    combinaciones = (
        rostros.groupby("imagen_original")["clase"]
        .apply(lambda clases: " + ".join(sorted(set(clases))))
    )

    imagenes = combinaciones.rename(
        "clases_presentes"
    ).reset_index()

    imagenes["grupo_division"] = imagenes[
        "clases_presentes"
    ].apply(
        lambda texto: (
            "contiene_incorrecta"
            if "mask_weared_incorrect" in texto
            else texto
        )
    )

    entrenamiento, temporal = train_test_split(
        imagenes,
        test_size=0.30,
        random_state=42,
        stratify=imagenes["grupo_division"],
    )

    validacion, prueba = train_test_split(
        temporal,
        test_size=0.50,
        random_state=42,
        stratify=temporal["grupo_division"],
    )

    return pd.concat([
        entrenamiento.assign(conjunto="entrenamiento"),
        validacion.assign(conjunto="validacion"),
        prueba.assign(conjunto="prueba"),
    ])


def guardar_recortes(rostros):
    registros = []

    for _, fila in rostros.iterrows():
        nombre = (
            f"{Path(fila['imagen_original']).stem}"
            f"_obj{int(fila['numero_objeto']):03d}.png"
        )

        carpeta = SALIDA / fila["conjunto"] / fila["clase"]
        carpeta.mkdir(parents=True, exist_ok=True)
        ruta_destino = carpeta / nombre

        caja = (
            int(fila["xmin_usado"]),
            int(fila["ymin_usado"]),
            int(fila["xmax_usado"]),
            int(fila["ymax_usado"]),
        )

        with Image.open(
            IMAGENES / fila["imagen_original"]
        ) as imagen:
            recorte = imagen.crop(caja).convert("RGBA")
            fondo = Image.new("RGBA", recorte.size, "white")

            recorte_final = Image.alpha_composite(
                fondo, recorte
            ).convert("RGB").resize(
                (128, 128),
                Image.Resampling.BILINEAR,
            )

            recorte_final.save(ruta_destino)

        registro = fila.to_dict()
        registro["ruta_recorte"] = str(ruta_destino)
        registros.append(registro)

    return pd.DataFrame(registros)


def main():
    SALIDA.mkdir(parents=True, exist_ok=True)

    anotaciones = leer_anotaciones()

    rostros = anotaciones[
        (anotaciones["ancho_rostro"] >= 10)
        & (anotaciones["alto_rostro"] >= 10)
    ].copy()

    asignaciones = dividir_por_imagen(rostros)

    rostros = rostros.merge(
        asignaciones[["imagen_original", "conjunto"]],
        on="imagen_original",
        how="left",
        validate="many_to_one",
    )

    if rostros["conjunto"].isna().any():
        raise ValueError("Hay rostros sin conjunto asignado")

    asignaciones.to_csv(
        SALIDA / "splits.csv", index=False
    )

    metadata = guardar_recortes(rostros)
    metadata.to_csv(
        SALIDA / "metadata.csv", index=False
    )

    print("Anotaciones originales:", len(anotaciones))
    print("Rostros excluidos por tamaño:", len(anotaciones) - len(rostros))
    print("Recortes generados:", len(metadata))
    print("Imágenes asignadas:", len(asignaciones))
    print(
        "Cajas conservadas con xmax corregido:",
        (metadata["xmax_original"] != metadata["xmax_usado"]).sum(),
    )
    print("\nRecortes por clase y conjunto:")
    print(pd.crosstab(metadata["clase"], metadata["conjunto"]))

metadata_csv = SALIDA / 'metadata.csv'
carpetas_listas = all((SALIDA / nombre).is_dir() for nombre in ('entrenamiento', 'validacion', 'prueba'))
if not metadata_csv.exists() or not carpetas_listas:
    main()
else:
    print('Recortes existentes: se reutilizan sin repetir el ETL.')

metadata = pd.read_csv(metadata_csv)
print('Recortes:', len(metadata))
print(pd.crosstab(metadata['clase'], metadata['conjunto']))
assert len(metadata) == 3603

## 3. Carga de datos y etiquetas

Las imágenes se cargan en lotes de 32. El orden de etiquetas se fija explícitamente para que los índices del modelo coincidan con la interpretación de las predicciones. La validación no se mezcla para conservar la correspondencia con las etiquetas reales.

In [ ]:
from pathlib import Path
import tensorflow as tf

DATOS = BASE / "data" / "processed"

CLASES = [
    "with_mask",
    "without_mask",
    "mask_weared_incorrect",
]

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATOS / "entrenamiento",
    class_names=CLASES,
    image_size=(128, 128),
    batch_size=32,
    label_mode="int",
    shuffle=True,
    seed=42,
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATOS / "validacion",
    class_names=CLASES,
    image_size=(128, 128),
    batch_size=32,
    label_mode="int",
    shuffle=False,
)

print("Orden de etiquetas:")
for numero, clase in enumerate(CLASES):
    print(numero, "=", clase)

for imagenes, etiquetas in train_ds.take(1):
    print("Forma del lote de imágenes:", imagenes.shape)
    print("Forma del lote de etiquetas:", etiquetas.shape)
    print("Valores de píxel:", imagenes.numpy().min(), "a", imagenes.numpy().max())

## 4. CNN básica

La primera red usa dos bloques `Conv2D + MaxPooling2D`, una capa densa de 32 neuronas y una salida `softmax` de tres clases. `Rescaling` convierte los píxeles de 0 a 255 a valores de 0 a 1. Se usa entropía cruzada categórica dispersa porque cada rostro tiene una sola etiqueta entera. Adam actualiza los parámetros a partir de los gradientes.

In [ ]:
tf.keras.utils.set_random_seed(42)

modelo_basico = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(128, 128, 3)),
    tf.keras.layers.Rescaling(1.0 / 255),

    tf.keras.layers.Conv2D(16, 3, activation="relu"),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Conv2D(32, 3, activation="relu"),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dense(3, activation="softmax"),
])

modelo_basico.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

modelo_basico.summary()

In [ ]:
historial_basico = modelo_basico.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
)

### Evaluación de la CNN básica en validación

Además de accuracy, se muestran precisión, recall y F1 de cada clase. El F1 macro da el mismo peso a las tres clases, lo cual importa porque `mask_weared_incorrect` tiene pocos ejemplos. La matriz indica dónde se equivoca la red.

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

etiquetas_reales = np.concatenate([
    etiquetas.numpy()
    for _, etiquetas in val_ds
])

probabilidades = modelo_basico.predict(val_ds)
etiquetas_predichas = np.argmax(probabilidades, axis=1)

print("Resultados en validación:")
print(
    classification_report(
        etiquetas_reales,
        etiquetas_predichas,
        labels=[0, 1, 2],
        target_names=CLASES,
        digits=4,
        zero_division=0,
    )
)

print("Matriz de confusión:")
print(
    confusion_matrix(
        etiquetas_reales,
        etiquetas_predichas,
        labels=[0, 1, 2],
    )
)

## 5. CNN con un bloque adicional

La segunda red conserva el mismo esquema y añade un tercer bloque convolucional de 64 filtros. Se entrena con el mismo número de épocas y el mismo conjunto para comparar las arquitecturas.

In [ ]:
tf.keras.utils.set_random_seed(42)

modelo_profundo = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(128, 128, 3)),
    tf.keras.layers.Rescaling(1.0 / 255),

    tf.keras.layers.Conv2D(16, 3, activation="relu"),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Conv2D(32, 3, activation="relu"),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Conv2D(64, 3, activation="relu"),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dense(3, activation="softmax"),
])

modelo_profundo.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

modelo_profundo.summary()

In [ ]:
historial_profundo = modelo_profundo.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
)

### Evaluación de la CNN con un bloque adicional

In [ ]:
probabilidades_profundo = modelo_profundo.predict(val_ds)
predicciones_profundo = np.argmax(
    probabilidades_profundo, axis=1
)

print("Modelo profundo: resultados en validación")
print(
    classification_report(
        etiquetas_reales,
        predicciones_profundo,
        labels=[0, 1, 2],
        target_names=CLASES,
        digits=4,
        zero_division=0,
    )
)

print("Matriz de confusión:")
print(
    confusion_matrix(
        etiquetas_reales,
        predicciones_profundo,
        labels=[0, 1, 2],
    )
)

## 6. Comparación y visualización

Se comparan accuracy y F1 macro en validación. Las curvas permiten observar si la pérdida de entrenamiento sigue bajando mientras la de validación deja de mejorar. Las matrices muestran los errores de cada clase.

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, recall_score

comparacion = pd.DataFrame([
    {
        'modelo': nombre,
        'accuracy_validacion': accuracy_score(etiquetas_reales, predichas),
        'f1_macro_validacion': f1_score(etiquetas_reales, predichas, average='macro'),
        'recall_mal_puesta': recall_score(etiquetas_reales, predichas, labels=[2], average=None)[0],
    }
    for nombre, predichas in [
        ('CNN básica', etiquetas_predichas),
        ('CNN profunda', predicciones_profundo),
    ]
])
print(comparacion.to_string(index=False, float_format=lambda n: f'{n:.4f}'))

In [ ]:
fig, axes = plt.subplots(
    2, 2,
    figsize=(11, 7),
    sharex=True,
    sharey="col",
    layout="constrained",
)

modelos = [
    ("CNN básica", historial_basico),
    ("CNN profunda", historial_profundo),
]

metricas = [
    ("loss", "val_loss", "Pérdida", "Entropía cruzada"),
    ("accuracy", "val_accuracy", "Accuracy", "Accuracy"),
]

for fila, (nombre_modelo, historial) in enumerate(modelos):
    epocas = range(1, len(historial.history["loss"]) + 1)

    for columna, (
        metrica_train,
        metrica_val,
        titulo,
        etiqueta_y,
    ) in enumerate(metricas):
        eje = axes[fila, columna]

        eje.plot(
            epocas,
            historial.history[metrica_train],
            color="#2878B5",
            marker="o",
            linestyle="--",
            label="Entrenamiento",
        )

        eje.plot(
            epocas,
            historial.history[metrica_val],
            color="#D95F02",
            marker="s",
            linestyle="-",
            label="Validación",
        )

        eje.set_title(f"{nombre_modelo} · {titulo}")
        eje.set_ylabel(etiqueta_y)
        eje.set_xticks(range(1, 11))
        eje.grid(alpha=0.2)
        eje.legend(loc="best")

for eje in axes[1]:
    eje.set_xlabel("Época")

ruta_curvas = (
    BASE / "outputs" / "figuras" / "curvas_comparativas.png"
)
fig.savefig(ruta_curvas, dpi=180, bbox_inches="tight")

plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix

matrices = [
    ("CNN básica", confusion_matrix(
        etiquetas_reales, etiquetas_predichas, labels=[0, 1, 2]
    )),
    ("CNN profunda", confusion_matrix(
        etiquetas_reales, predicciones_profundo, labels=[0, 1, 2]
    )),
]

nombres = ["Bien puesta", "Sin mascarilla", "Mal puesta"]

fig = plt.figure(figsize=(14, 5), layout="constrained")
espacios = fig.add_gridspec(
    1, 3,
    width_ratios=[1, 1, 0.04],
    wspace=0.18,
)

axes = [
    fig.add_subplot(espacios[0, 0]),
    fig.add_subplot(espacios[0, 1]),
]
eje_color = fig.add_subplot(espacios[0, 2])

for eje, (titulo, matriz) in zip(axes, matrices):
    porcentajes = matriz / matriz.sum(axis=1, keepdims=True)

    imagen = eje.imshow(
        porcentajes, cmap="Blues", vmin=0, vmax=1
    )

    for fila in range(3):
        for columna in range(3):
            porcentaje = porcentajes[fila, columna]

            eje.text(
                columna, fila,
                f"{matriz[fila, columna]}\n{porcentaje:.1%}",
                ha="center",
                va="center",
                fontsize=11,
                color="white" if porcentaje > 0.5 else "black",
            )

    eje.set_title(titulo)
    eje.set_xticks(range(3), nombres, rotation=20, ha="right")
    eje.set_yticks(range(3), nombres)
    eje.set_xlabel("Clase predicha")
    eje.set_ylabel("Clase real")

barra = fig.colorbar(imagen, cax=eje_color)
barra.set_label("Proporción de la clase real")

fig.suptitle("Matrices de confusión en validación", fontsize=14)

ruta_figura = BASE / "outputs" / "figuras" / "matrices_validacion.png"
fig.savefig(ruta_figura, dpi=180, bbox_inches="tight")
plt.show()

**Lectura de la corrida local documentada en el paper:** la CNN profunda tuvo mayor accuracy de validación (0.9541 frente a 0.9468), pero la básica tuvo mayor F1 macro (0.7725 frente a 0.7472) y detectó 7 de 16 casos de mascarilla mal puesta, frente a 4 de 16. Se eligió provisionalmente la CNN básica para la interfaz. En Colab las cifras pueden variar ligeramente según el entorno; interpreta los valores que produzca esta ejecución antes de sacar una conclusión nueva. No se usó el conjunto de prueba para elegir el modelo.

## 7. Guardar los modelos

Se guardan ambos modelos en formato Keras. `app.py` utiliza `models/cnn_basica.keras`, que corresponde a la elección documentada en el reporte. Si ejecutas esto en Colab, los archivos quedan en `/content/face_mask_detection/models` y puedes descargarlos desde el panel de archivos.

In [ ]:
carpeta_modelos = BASE / "models"
carpeta_modelos.mkdir(parents=True, exist_ok=True)

modelo_basico.save(carpeta_modelos / "cnn_basica.keras")
modelo_profundo.save(carpeta_modelos / "cnn_profunda.keras")

print("Básico:", (carpeta_modelos / "cnn_basica.keras").exists())
print("Profundo:", (carpeta_modelos / "cnn_profunda.keras").exists())

## 8. Una predicción de ejemplo

Se toma un recorte de validación para mostrar la salida de las tres clases. Esta demostración no sustituye las métricas calculadas sobre todas las observaciones.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

for imagenes, etiquetas in val_ds.take(1):
    rostro = imagenes[0]
    etiqueta_real = int(etiquetas[0])
    break

valores = modelo_basico.predict(rostro[None, ...], verbose=0)[0]
print('Clase real:', CLASES[etiqueta_real])
print('Clase predicha:', CLASES[int(np.argmax(valores))])
for clase, valor in zip(CLASES, valores):
    print(f'{clase}: {valor:.1%}')
plt.imshow(rostro.numpy().astype('uint8'))
plt.axis('off')
plt.show()

## Referencias

- [Dataset Face Mask Detection, Kaggle](https://www.kaggle.com/datasets/andrewmvd/face-mask-detection/data)
- [Tutorial de clasificación de imágenes, TensorFlow](https://www.tensorflow.org/tutorials/images/classification)
- [Documentación de `kagglehub`](https://github.com/Kaggle/kagglehub)